<a href="https://colab.research.google.com/github/EstherMan05/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-14%20%E2%80%94%20Mini-ETL%20on%20Walmart%20Sample%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Mini-ETL on Walmart Sample

**Studio — 2026-10-14 · Fall 2026**  
**Class time:** 45 minutes

---

## The midterm, in miniature

Twelve rows of the real thing. Every problem in this sample is in the full dataset you get Monday, just at a scale where you can still see the answer.

You are writing the whole pipeline as **functions**, not as loose cells. That is not stylistic — Monday you will run the same logic on a much larger file, and a pipeline built from functions is one you can re-run and hand to a teammate. A pipeline built from cells in a specific click order is not.

The extract stage is worked below as an example. Transform and load are yours to build.

In [1]:
import pandas as pd, sqlite3
from io import StringIO

RAW = '''transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1,3.86
TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1,5.00
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3,$4.11
TXN5,fl-239,2024-09-06T12:15:00,WATER-24,bottled water,grocery,4,$5.00
TXN6,FL-105,2024-09-07 08:02:00,BTL-WATER-24,Bottled Water 24pk,Grocery,6,5.00
TXN7,FL-105,2024-09-07T09:45:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,5,4.11
TXN8,FL-239,,PT-12,pop-tarts strawberry,Breakfast,2,$4.11
TXN9,FL-105,2024-09-07 14:20:00,FLASH-AA,Flashlight AA,Hardware,NULL,12.99
TXN10,FL-239,2024-09-07T15:00:00,POPTART-BLUE,Pop-Tarts Blueberry,Breakfast,1,4.11
TXN11,FL-105,09/07/2024 16:10,SKU#459812,Strawberry Pop-Tart,breakfast,4,4.11'''

### Extract — worked example

The extract function does one job: get the raw data in, and report what arrived. It does **not** clean anything. Keeping the stages separate is what lets you answer "was this a source problem or did I do it?" later.

In [2]:
def extract():
    """Read the raw transactions. No cleaning here -- report only."""
    df = pd.read_csv(StringIO(RAW))
    print(f'extracted {len(df)} rows, {len(df.columns)} columns')
    print('dtypes that arrived as text:',
          [c for c in df.columns if df[c].dtype == object])
    print('nulls:', df.isnull().sum().sum())
    print('exact duplicates:', df.duplicated().sum())
    return df

raw = extract()
raw.head()

extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
1,TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1.0,3.86
2,TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1.0,5.00
3,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
4,TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3.0,$4.11


Note what the report already told us: `quantity` and `unit_price` both arrived as text, there is a null, and there is a duplicate. That is your transform to-do list, generated from the data instead of from guessing.

**TODO:** before writing any transform, list the specific problems you can see in these twelve rows. There are at least six.

1. TXN1 appears twice, as an exact duplicate.
2. unit_price is text, and some values carry a $ ($4.11 vs 4.11).
3. store_id has several spellings of two stores (FL-105, fl-105, FL 239, fl-239)
4. timestamp comes in three formats: ISO with a space, ISO with T, and MM/DD/YYYY
5. TXN8 has no timestamp
6. TXN9 has no quantity

### Build 1 — Transform: types and duplicates

**TODO:** write `transform_types(df)` that returns a new frame with:

- exact duplicate rows removed
- `unit_price` as a float
- `quantity` numeric, with a stated decision about the null and the negative
- `store_id` normalized — note the four spellings of the same two stores
- `timestamp` as a real datetime, coercing the empty one

It must not modify the frame passed in. Print a before/after row count.

In [3]:
def transform_types(df):
    out = df.copy()                                   # never touch the frame passed in
    out = out.drop_duplicates()                       # exact duplicates (TXN1)

    out['unit_price'] = (out['unit_price'].astype(str)
                         .str.replace('$', '', regex=False).str.strip().astype(float))

    # Decision: null qty stays NaN (unknown, not zero); negative qty stays (a refund
    # is a real event that reduces units and revenue). Neither row is dropped.
    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')

    out['store_id'] = (out['store_id'].str.upper().str.strip()
                       .str.replace(r'[\s_]+', '-', regex=True))   # 'FL 239' -> 'FL-239'

    out['category'] = out['category'].str.strip().str.title()

    out['timestamp'] = pd.to_datetime(out['timestamp'], errors='coerce', format='mixed')
    return out.reset_index(drop=True)

typed = transform_types(raw)
print('rows:', len(raw), '->', len(typed))
typed.dtypes

rows: 12 -> 11


,0
transaction_id,object
store_id,object
timestamp,datetime64[ns]
sku,object
product_name,object
category,object
quantity,float64
unit_price,float64


### Build 2 — Transform: consolidate the products

Four SKUs are strawberry Pop-Tarts. One is blueberry — leave it alone, and be ready to say why. Two are the same bottled water.

**TODO:** write `consolidate(df)` that adds a `canonical_sku` column. Any SKU not in your mapping should keep its original value, not become `NaN`.

Then print units per canonical SKU and confirm the total quantity did not change — consolidation renames things, it never adds or removes units.

In [4]:
CANON = {
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'WATER-24': 'WATER-24',
    'BTL-WATER-24': 'WATER-24',
}

def consolidate(df):
    out = df.copy()
    out['canonical_sku'] = out['sku'].map(CANON).fillna(out['sku'])   # unmapped keep themselves
    return out

consolidated = consolidate(typed)
assert consolidated['quantity'].sum() == typed['quantity'].sum(), 'consolidation changed total units'
print(consolidated.groupby('canonical_sku')['quantity'].sum())

canonical_sku
FLASH-AA          0.0
POPTART-BLUE      1.0
POPTART-STRAW    17.0
WATER-24          9.0
Name: quantity, dtype: float64


**Why blueberry stays separate:** _..._

### Build 3 — Transform: derive what the analysis needs

**TODO:** add `revenue` (quantity times unit price) and a `date` column from the timestamp. The `date` column is what you will join weather onto next week, so it needs to be a date — not a full timestamp, not a string.

In [5]:
def derive(df):
    out = df.copy()
    out['revenue'] = out['quantity'] * out['unit_price']
    out['date'] = out['timestamp'].dt.normalize()     # stays datetime64, so it joins to weather
    return out

enriched = derive(consolidated)
enriched[['transaction_id', 'canonical_sku', 'quantity', 'revenue', 'date']]

,transaction_id,canonical_sku,quantity,revenue,date
0,TXN1,POPTART-STRAW,2.0,8.22,2024-09-06
1,TXN2,POPTART-STRAW,1.0,3.86,2024-09-06
2,TXN3,WATER-24,-1.0,-5.00,2024-09-06
3,TXN4,POPTART-STRAW,3.0,12.33,2024-09-06
4,TXN5,WATER-24,4.0,20.00,2024-09-06
5,TXN6,WATER-24,6.0,30.00,2024-09-07
6,TXN7,POPTART-STRAW,5.0,20.55,2024-09-07
7,TXN8,POPTART-STRAW,2.0,8.22,NaT
8,TXN9,FLASH-AA,NaN,NaN,2024-09-07
9,TXN10,POPTART-BLUE,1.0,4.11,2024-09-07


### Build 4 — Load, and verify the load

**TODO:** write `load(df, conn)` that writes the clean frame to a `clean_sales` table, then reads back the row count and asserts it matches. Writing without verifying is how you discover at 11pm that `if_exists='append'` doubled your table.

In [6]:
def load(df, conn, table='clean_sales'):
    df.to_sql(table, conn, index=False, if_exists='replace')
    n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {table}', conn)['n'].iloc[0]
    assert n == len(df), f'load mismatch: wrote {len(df)}, table has {n}'
    print(f'loaded {n} rows into {table} (verified)')

conn = sqlite3.connect(':memory:')
load(enriched, conn)

loaded 11 rows into clean_sales (verified)


### Build 5 — run the whole thing as one pipeline

**TODO:** wire the stages together in one function, then run it twice against a fresh connection and confirm you get identical results. That is the idempotency property from Monday, tested rather than assumed.

In [7]:
def run_pipeline(conn):
    df = extract()
    df = transform_types(df)
    df = consolidate(df)
    df = derive(df)
    load(df, conn)
    return df

first = run_pipeline(sqlite3.connect(':memory:'))
second = run_pipeline(sqlite3.connect(':memory:'))

assert first.shape == second.shape
assert round(first['revenue'].sum(), 2) == round(second['revenue'].sum(), 2)
print('rows:', first.shape, '| revenue:', round(first['revenue'].sum(), 2))

extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1
loaded 11 rows into clean_sales (verified)
extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1
loaded 11 rows into clean_sales (verified)
rows: (11, 11) | revenue: 118.73


### Build 6 — the question the pipeline exists to answer

**TODO:** did strawberry Pop-Tart sales rise from Sept 6 to Sept 7? Print units by date for your canonical strawberry SKU, and say whether two days is enough evidence to conclude anything.

That second part matters. On the midterm you will have a real baseline to compare against; here you do not, and saying so is the correct answer.

In [8]:
straw = enriched[enriched['canonical_sku'] == 'POPTART-STRAW']
print(straw.groupby('date')['quantity'].sum())
print('strawberry units with no date:', straw.loc[straw['date'].isna(), 'quantity'].sum())

date
2024-09-06    6.0
2024-09-07    9.0
Name: quantity, dtype: float64
strawberry units with no date: 2.0


---

## Checkpoint (participation)

Report your row count after transform, your canonical SKU count, and whether running the pipeline twice gave the same revenue total.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [9]:
rows_after_transform = len(typed)
canonical_skus = enriched['canonical_sku'].nunique()
idempotent = (first.shape == second.shape and
              round(first['revenue'].sum(), 2) == round(second['revenue'].sum(), 2))
blueberry_call = 'Blueberry is a different product, not another code for strawberry, so merging it would erase a real distinction.'

print('rows after transform:', rows_after_transform)
print('canonical SKUs:', canonical_skus)
print('same result twice:', idempotent)
print('blueberry:', blueberry_call)

rows after transform: 11
canonical SKUs: 4
same result twice: True
blueberry: Blueberry is a different product, not another code for strawberry, so merging it would erase a real distinction.
